# agentlab voice: talk to Gemma 4 on free Colab
Runtime -> Change runtime type -> **T4 GPU**, then Run all. English only.

Pipeline: mic -> Silero VAD + faster-whisper -> Gemma 4 E4B (4-bit) -> Kokoro voice.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/RUTURAJROCks/AgentLab-Experiments.git
%cd AgentLab-Experiments
!pip -q install -U -r voice/requirements.txt
!apt-get -qq install -y espeak-ng > /dev/null

Optional: add `HF_TOKEN` in Colab Secrets (key icon) for faster model downloads.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
MODE = 'hybrid'   # 'native', 'cascade' or 'hybrid'
from voice.agent import VoiceAgent
agent = VoiceAgent(mode=MODE)

## 1. Quick text check

In [ ]:
print(agent.reply(text='Give me one tip to sleep better tonight.', keep_history=False)[0])

## 2. Talk to it
Open the public link, allow the mic, speak, press stop. The reply plays as it is generated.

In [ ]:
from voice.app import build
build(agent).queue().launch(share=True, debug=False)

## 3. Benchmark on VoiceBench (official scorers)
100 questions per subset takes roughly 20 to 60 minutes on a T4. Raise `--limit` for full runs.

In [ ]:
!git clone -q https://github.com/MatthewCYM/VoiceBench.git
del agent
import gc, torch; gc.collect(); torch.cuda.empty_cache()
!python -m voice.bench --mode {MODE} --subsets openbookqa ifeval advbench bbh --limit 100

## 4. Compare modes
Run the cell above again with MODE = 'native' and 'cascade', then compare `report-*.json`. That table is the X thread.

In [ ]:
import json, glob
for f in sorted(glob.glob('report-*.json')):
    print(f, json.dumps(json.load(open(f)), indent=1))